In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf
!pip install transformers
!pip install accelerate
!pip install bitsandbytes

In [ ]:
!pip install langchain-core

In [ ]:
from huggingface_hub import login
login()  # paste your HuggingFace token here

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader

docs = []

for file_name in uploaded.keys():
    ext = os.path.splitext(file_name)[1].lower()

    if ext == ".txt":
        print(f"📄 TXT detected: {file_name}")
        loader = TextLoader(file_name, encoding="utf-8")

    elif ext == ".pdf":
        print(f"📕 PDF detected: {file_name}")
        loader = PyPDFLoader(file_name)

    else:
        print(f"❌ Skipping: {file_name}")
        continue

    docs.extend(loader.load())

print("Total Documents:", len(docs))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print("Total Chunks:", len(chunks))
print("\n--- Sample Chunk ---")
print(chunks[0].page_content)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"}  # use "cpu" if no GPU
)

print("✅ Embedding model loaded")

In [ ]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("✅ FAISS vector store created")

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # return top 3 relevant chunks
)

print("✅ Retriever ready")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "google/gemma-2-2b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ Gemma-2 loaded")

In [ ]:
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,        # ✅ required when using temperature
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id  # ✅ stops padding warning too
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM pipeline ready")

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful assistant. Use the context below to answer the question.
If you don't know the answer from the context, just say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

print("✅ Prompt template ready")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Prompt ──────────────────────────────────────────────────
template = """
You are a helpful assistant. Use the context below to answer the question.
If you don't know the answer from the context, just say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# ── Format retrieved docs into single string ─────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── Build RAG Chain ──────────────────────────────────────────
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain ready!")

In [ ]:
def ask(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)
    result = rag_chain.invoke(question)
    print(f"💡 Answer:\n{result}")

# Test it
ask("diffrence between react and angular?")